# Day 6 — confirmatory analysis

Runs the pre-registered tests **in the order they were registered**, each against its stated
threshold, and prints HOLDS or FAILS. Exploratory analyses come after an explicit divider and are
listed before any of them is run.

No GPU needed. Inputs: `results.csv`, `corpus.csv`, the rating files, `audit_llm.jsonl`,
`rating_key_SEALED.csv`, `graph_texts.jsonl`.

| Part | Test | Pre-registered threshold |
|---|---|---|
| 2 | Data integrity | 255 rows, statuses as recorded |
| 3 | Rater reliability | duplicate kappa; P5 uninterpretable below 0.6 |
| 4 | **P2** replacement does not discriminate | range < 0.15 **and** eta² < 0.20 |
| 5 | **P4** completeness even less discriminative | completeness range < replacement range |
| 6 | **P3** error location discriminates | eta²(normalised interior Gini) ≥ 2 × eta²(logit replacement) |
| 7 | **P1** category ranking | Spearman, predicted vs observed, 8 categories |
| 8 | **P5** human ratings | \|rho\| < 0.4, gated on duplicate kappa ≥ 0.6 |
| 9 | Falsification (section 5) | eta² < 0.10 everywhere; LOCO R² ≤ 0 |
| 10 | Three regression models (6.1) | category / continuous / both, VIF, Benjamini–Hochberg |

**The order is not cosmetic.** Once an exploratory result has been seen it is hard to unsee when
writing up a confirmatory one, so every registered test runs first.

## Part 1 — Load

In [8]:
# Build audit_llm.jsonl from saved reply files (no model needed)
import glob, json, os

texts = {json.loads(l)["presentation"]: json.loads(l)
         for l in open(find_input("graph_texts.jsonl")) if l.strip()}
h1i = (pd.DataFrame([json.loads(l) for l in open(find_input("ratings_js.jsonl")) if l.strip()])
         .drop_duplicates("presentation").set_index("presentation"))

LLM_MODEL = "claude-opus-4.5 (private chat, batches of 10)"   # whatever you used

arecs = []
for path in sorted(glob.glob("/kaggle/input/**/reply_audit_*.json", recursive=True)):
    for d in json.load(open(path)):
        n = int(d["presentation"])
        arecs.append(dict(presentation=n, prompt_id=texts[n]["prompt_id"],
                          human_rating=int(h1i.loc[n].rating), model=LLM_MODEL,
                          reason_supported=bool(d["reason_supported"]),
                          rating_follows=bool(d["rating_follows"]),
                          your_rating=int(d["your_rating"]), comment=d.get("comment", "")))

assert arecs, "no reply_audit_*.json found — is the dataset attached?"
with open(f"{OUT}/audit_llm.jsonl", "w") as f:
    for r in arecs:
        f.write(json.dumps(r) + "\n")
print(f"{len(arecs)} audit records written to {OUT}/audit_llm.jsonl")

50 audit records written to /kaggle/working/out/audit_llm.jsonl


In [44]:
import glob, json
man = pd.read_csv(find_input("rating_manifest.csv"))
pid_of = dict(zip(man.presentation, man.prompt_id))

def build(pattern, rater, model, outfile):
    recs = []
    for path in sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)):
        for d in json.load(open(path)):
            n = int(d["presentation"])
            recs.append(dict(presentation=n, prompt_id=pid_of[n], rater=rater,
                             rating=int(d["rating"]), reason=d.get("reason", ""),
                             model=model))
    assert recs, f"no files matching {pattern}"
    seen = {r["presentation"] for r in recs}
    print(f"{rater}: {len(recs)} ratings, missing {sorted(set(pid_of) - seen) or 'none'}")
    with open(f"{OUT}/{outfile}", "w") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")

# build("reply_rate_assistant*.json", "assistant", "assistant-with-context", "ratings_assistant.jsonl")
# and if ratings_llm.jsonl is missing too:
build("reply_rate_?.json", "llm_opus", "claude-opus-4.5 (private chat)", "ratings_llm.jsonl")

llm_opus: 50 ratings, missing none


In [58]:
import os, json, glob, itertools
import numpy as np, pandas as pd
from scipy import stats

OUT = "/kaggle/working/out"; os.makedirs(OUT, exist_ok=True)
pd.set_option("display.width", 140)

def find_input(name, required=True):
    hits = (glob.glob(f"/kaggle/input/**/{name}", recursive=True)
            + glob.glob(f"/kaggle/working/**/{name}", recursive=True))
    if not hits and required:
        raise FileNotFoundError(name)
    return hits[0] if hits else None

res = pd.read_csv(find_input("results.csv"), keep_default_na=False)
corpus = pd.read_csv(find_input("corpus.csv"),
                     dtype={"expected": str, "subtype": str, "prompt": str}, keep_default_na=False)
for col in ["replacement_score", "completeness_score", "error_mass_total", "err_pos_gini_norm",
            "err_final_share", "mean_path_length", "err_late_layers", "err_pos_argmax_norm"]:
    if col in res.columns:
        res[col] = pd.to_numeric(res[col], errors="coerce")

df = res[res.status == "ok"].merge(
    corpus[["prompt_id", "n_tok", "next_token_entropy", "top1_prob", "mean_token_logid",
            "task_ok", "subtype"]], on="prompt_id", how="left", suffixes=("", "_c"))
print(f"{len(res)} rows, {len(df)} with status ok")
print(df.groupby("category").size().to_string())

255 rows, 255 with status ok
category
arithmetic              28
code_completion         29
entity_known_unknown    32
factual_recall          40
induction               24
multi_hop               32
obfuscated              44
syntactic_agreement     26


In [65]:
def load_ratings(path, rater):
    if path is None or not os.path.exists(path):
        return None
    d = pd.DataFrame([json.loads(l) for l in open(path) if l.strip()])
    d["rater"] = rater
    return d.dropna(subset=["rating"]) if "rating" in d.columns else d

STREAMS = {}
for rater, fname in [("human", "ratings_js.jsonl"),
                     ("llm_opus", "ratings_llm.jsonl"),
                     ("assistant", "ratings_assistant.jsonl")]:
    d = load_ratings(find_input(fname, required=(rater == "human")), rater)
    if d is not None and len(d):
        STREAMS[rater] = d
print({k: len(v) for k, v in STREAMS.items()})

manifest = pd.read_csv(find_input("rating_manifest.csv"),
                       dtype={"prompt": str, "subtype": str}, keep_default_na=False)
key = pd.read_csv(find_input("rating_key_SEALED.csv"))          # bins; opened only now
audit = load_ratings(find_input("audit_llm.jsonl", required=False), "audit")
print("audit records:", 0 if audit is None else len(audit))

{'human': 53, 'llm_opus': 50, 'assistant': 50}
audit records: 50


In [66]:
import json
rows = [json.loads(l) for l in open(f"{OUT}/ratings_js.jsonl") if l.strip()]
STREAMS["human"] = pd.DataFrame(rows).assign(rater="human")
print(len(STREAMS["human"]), "rows;",
      (STREAMS["human"].prompt_id.value_counts() == 2).sum(), "pairs")

50 rows; 10 pairs


## Part 2 — Data integrity

In [67]:
print(pd.crosstab(res.category, res.status, margins=True).to_string())
print(f"\nmissing prompts: {sorted(set(corpus.prompt_id) - set(res.prompt_id)) or 'none'}")
print(f"BOS structural-zero violations: {(res.status == 'structural_zero_violated').sum()}")
print(f"metrics retried on CPU: {(res.get('metrics_device') == 'cpu').sum()}")

NOISE_MAX = 3.4e-4          # D3.5: maximum re-attribution difference
print(f"\nnoise floor in use: median 9.0e-05, max {NOISE_MAX:.1e}; "
      f"category differences below ~1e-3 are treated as not meaningful (6.4 as revised)")

status                 ok  All
category                      
arithmetic             28   28
code_completion        29   29
entity_known_unknown   32   32
factual_recall         40   40
induction              24   24
multi_hop              32   32
obfuscated             44   44
syntactic_agreement    26   26
All                   255  255

missing prompts: none
BOS structural-zero violations: 0
metrics retried on CPU: 0

noise floor in use: median 9.0e-05, max 3.4e-04; category differences below ~1e-3 are treated as not meaningful (6.4 as revised)


## Part 3 — Rater reliability

Quadratic weighted kappa on the 10 unmarked duplicates, per stream, then pairwise agreement on
first presentations. Per D3.11 the assistant stream's duplicate figure is **not** comparable: its
duplicate pairs shared a context window.

In [68]:
def quadratic_kappa(x, y, k=3):
    x, y = np.asarray(x, int), np.asarray(y, int)
    n = len(x)
    O = np.zeros((k, k))
    for i, j in zip(x, y):
        O[i, j] += 1
    W = np.array([[(i - j) ** 2 / (k - 1) ** 2 for j in range(k)] for i in range(k)])
    E = np.outer(np.bincount(x, minlength=k), np.bincount(y, minlength=k)) / n
    den = (W * E).sum()
    return float(1 - (W * O).sum() / den) if den > 0 else float("nan")

REL = {}
for rater, d in STREAMS.items():
    dup = d.groupby("prompt_id").filter(lambda x: len(x) == 2)
    pr = dup.sort_values("presentation").groupby("prompt_id").rating.apply(list)
    if len(pr) == 0:
        continue
    a = [p[0] for p in pr]; b = [p[1] for p in pr]
    REL[rater] = dict(n_pairs=len(a), exact=float(np.mean(np.array(a) == np.array(b))),
                      kappa=quadratic_kappa(a, b))
    note = "  (NOT comparable: shared context)" if rater == "assistant" else ""
    print(f"{rater:10s} pairs={len(a):2d}  exact={REL[rater]['exact']:.0%}  "
          f"kappa={REL[rater]['kappa']:+.3f}{note}")

HUMAN_KAPPA = REL.get("human", {}).get("kappa", float("nan"))
print(f"\nP5 gate: human duplicate kappa {HUMAN_KAPPA:.3f} "
      f"{'>= 0.6, P5 interpretable' if HUMAN_KAPPA >= 0.6 else '< 0.6, P5 UNINTERPRETABLE (section 5)'}")

human      pairs=10  exact=90%  kappa=+0.889
llm_opus   pairs=10  exact=80%  kappa=+0.762
assistant  pairs=10  exact=100%  kappa=+1.000  (NOT comparable: shared context)

P5 gate: human duplicate kappa 0.889 >= 0.6, P5 interpretable


In [77]:
firsts = {r: d.sort_values("presentation").drop_duplicates("prompt_id")
                 .set_index("prompt_id").rating.astype(int)
          for r, d in STREAMS.items()}
rows = []
for a, b in itertools.combinations(firsts, 2):
    x, y = firsts[a].align(firsts[b], join="inner")
    rows.append(dict(pair=f"{a} vs {b}", n=len(x),
                     exact=round(float(np.mean(x.values == y.values)), 3),
                     kappa=round(quadratic_kappa(x.values, y.values), 3)))
PAIRWISE = pd.DataFrame(rows)
print(PAIRWISE.to_string(index=False))
print("\ndistribution by stream:")
print(pd.DataFrame({r: s.value_counts().reindex([0, 1, 2]).fillna(0).astype(int)
                    for r, s in firsts.items()}).to_string())
print("\nNote (D3.11): the LLM streams were given the human rater's written rubric and the "
      "assistant stream co-developed it, so agreement is not independent convergence.")

                 pair  n  exact  kappa
    human vs llm_opus 40  0.675  0.677
   human vs assistant 40  0.625  0.674
llm_opus vs assistant 40  0.800  0.818

distribution by stream:
        human  llm_opus  assistant
rating                            
0          17        21         27
1          17        14          6
2           6         5          7

Note (D3.11): the LLM streams were given the human rater's written rubric and the assistant stream co-developed it, so agreement is not independent convergence.


## Part 4 — P2 (primary)

*Replacement score will not discriminate between categories*: between-category range of means
**under 0.15**, and category explains **under 20%** of variance.

Variance explained is eta² from a one-way ANOVA on the **logit-transformed** score, as section 6.1
specifies for a bounded proportion. The untransformed value is printed alongside.

In [70]:
def logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, float), eps, 1 - eps)
    return np.log(p / (1 - p))

def eta2(values, groups):
    v = np.asarray(values, float); g = np.asarray(groups)
    grand = v.mean()
    ss_tot = ((v - grand) ** 2).sum()
    ss_bet = sum(len(v[g == k]) * (v[g == k].mean() - grand) ** 2 for k in np.unique(g))
    return float(ss_bet / ss_tot) if ss_tot > 0 else float("nan")

cat = df.category.values
rep, repl = df.replacement_score.values, logit(df.replacement_score)

means = df.groupby("category").replacement_score.agg(["mean", "std", "min", "max", "size"])
print(means.round(4).sort_values("mean").to_string())

P2_RANGE = float(means["mean"].max() - means["mean"].min())
P2_ETA2  = eta2(repl, cat)
P2_ETA2_RAW = eta2(rep, cat)
print(f"\nbetween-category range of means : {P2_RANGE:.4f}   (threshold < 0.15)")
print(f"eta2, logit-transformed         : {P2_ETA2:.4f}   (threshold < 0.20)")
print(f"eta2, untransformed             : {P2_ETA2_RAW:.4f}")
print(f"range vs noise floor            : {P2_RANGE / 3.4e-4:.0f}x the maximum re-attribution difference")

P2 = (P2_RANGE < 0.15) and (P2_ETA2 < 0.20)
print(f"\n>>> P2 {'HOLDS' if P2 else 'FAILS'}")

                        mean     std     min     max  size
category                                                  
arithmetic            0.7092  0.0129  0.6936  0.7381    28
obfuscated            0.7273  0.0178  0.6804  0.7575    44
multi_hop             0.7279  0.0170  0.6972  0.7644    32
code_completion       0.7357  0.0235  0.6850  0.7642    29
factual_recall        0.7381  0.0243  0.7017  0.7779    40
syntactic_agreement   0.7414  0.0255  0.6925  0.7817    26
entity_known_unknown  0.7423  0.0079  0.7324  0.7565    32
induction             0.7455  0.0124  0.7212  0.7630    24

between-category range of means : 0.0363   (threshold < 0.15)
eta2, logit-transformed         : 0.2383   (threshold < 0.20)
eta2, untransformed             : 0.2418
range vs noise floor            : 107x the maximum re-attribution difference

>>> P2 FAILS


In [71]:
h = STREAMS["human"]
counts = h.prompt_id.value_counts()
print("prompt_ids with 2 ratings:", (counts == 2).sum())
print("prompt_ids with 1 rating:", (counts == 1).sum())

llm_pairs = set(STREAMS["llm_opus"].prompt_id.value_counts().pipe(lambda s: s[s == 2]).index)
human_pairs = set(counts[counts == 2].index)
print("pairs the LLM has but you don't:", llm_pairs - human_pairs)

prompt_ids with 2 ratings: 10
prompt_ids with 1 rating: 30
pairs the LLM has but you don't: set()


In [72]:
raw = pd.DataFrame([json.loads(l) for l in open(find_input("ratings_js_RAW.jsonl"))])
print("raw pairs:", (raw.drop_duplicates('presentation').prompt_id.value_counts() == 2).sum())

raw pairs: 10


In [73]:
raw = pd.DataFrame([json.loads(l) for l in open(find_input("ratings_js_RAW.jsonl"))])
print("raw pairs:", (raw.drop_duplicates('presentation').prompt_id.value_counts() == 2).sum())

raw pairs: 10


In [74]:
raw = pd.DataFrame([json.loads(l) for l in open(find_input("ratings_js_RAW.jsonl"))])
cur = pd.DataFrame([json.loads(l) for l in open(find_input("ratings_js.jsonl"))])
lost = set(raw.presentation) - set(cur.presentation)
print("presentations lost:", sorted(lost))
print(raw[raw.presentation.isin(lost)][["presentation", "prompt_id", "rating", "at"]].to_string(index=False))
print("\nraw rows:", len(raw), "current rows:", len(cur))

presentations lost: []
Empty DataFrame
Columns: [presentation, prompt_id, rating, at]
Index: []

raw rows: 53 current rows: 53


In [75]:
import json
raw = [json.loads(l) for l in open(find_input("ratings_js.jsonl")) if l.strip()]
seen, kept = set(), []
for r in raw:
    k = int(r["presentation"])
    if k not in seen:
        seen.add(k); kept.append(r)

with open(f"{OUT}/ratings_js_RAW.jsonl", "w") as f:
    for r in raw: f.write(json.dumps(r) + "\n")
with open(f"{OUT}/ratings_js.jsonl", "w") as f:
    for r in kept: f.write(json.dumps(r) + "\n")

import pandas as pd
k = pd.DataFrame(kept)
print(f"{len(raw)} -> {len(kept)} rows; "
      f"{(k.prompt_id.value_counts() == 2).sum()} duplicate pairs; "
      f"unique presentations: {k.presentation.nunique()}")

53 -> 50 rows; 10 duplicate pairs; unique presentations: 50


## Part 5 — P4

*Completeness is even less discriminative than replacement*: smaller between-category range.

In [78]:
cmeans = df.groupby("category").completeness_score.agg(["mean", "min", "max"])
print(cmeans.round(4).sort_values("mean").to_string())
P4_RANGE = float(cmeans["mean"].max() - cmeans["mean"].min())
P4_ETA2 = eta2(logit(df.completeness_score), cat)
print(f"\ncompleteness range {P4_RANGE:.4f}  vs  replacement range {P2_RANGE:.4f}")
print(f"completeness eta2  {P4_ETA2:.4f}  vs  replacement eta2  {P2_ETA2:.4f}")
P4 = P4_RANGE < P2_RANGE
print(f"\n>>> P4 {'HOLDS' if P4 else 'FAILS'}")

                        mean     min     max
category                                    
arithmetic            0.9277  0.9217  0.9363
multi_hop             0.9278  0.9179  0.9396
factual_recall        0.9305  0.9200  0.9407
obfuscated            0.9309  0.9161  0.9384
syntactic_agreement   0.9325  0.9211  0.9421
induction             0.9332  0.9234  0.9403
code_completion       0.9341  0.9195  0.9415
entity_known_unknown  0.9356  0.9323  0.9394

completeness range 0.0079  vs  replacement range 0.0363
completeness eta2  0.2099  vs  replacement eta2  0.2383

>>> P4 HOLDS


## Part 6 — P3

*Error location discriminates where error quantity does not*: eta² of category on the **normalised
interior Gini** (D3.7, the single pre-specified primary measure) is at least **2×** eta² on
logit-transformed replacement. The other six location summaries are secondary and do not count.

In [79]:
P3_PRIMARY = eta2(df.err_pos_gini_norm.values, cat)
print(f"eta2, normalised interior Gini (primary) : {P3_PRIMARY:.4f}")
print(f"eta2, logit replacement                  : {P2_ETA2:.4f}")
print(f"ratio                                    : {P3_PRIMARY / max(P2_ETA2, 1e-12):.2f}x  (threshold >= 2)")
P3 = P3_PRIMARY >= 2 * P2_ETA2
print(f"\n>>> P3 {'HOLDS' if P3 else 'FAILS'}")

print("\nsecondary location summaries (reported, not counted toward P3):")
sec = {}
for col in ["err_final_share", "err_pos_argmax_norm", "err_late_layers", "error_mass_total",
            "mean_path_length", "err_pos_gini"]:
    if col in df.columns and df[col].notna().any():
        sec[col] = eta2(df[col].values, cat)
print(pd.Series(sec).round(4).sort_values(ascending=False).to_string())
print(f"\nby category, primary measure:")
print(df.groupby("category").err_pos_gini_norm.agg(["mean", "std"]).round(3)
        .sort_values("mean").to_string())

eta2, normalised interior Gini (primary) : 0.4441
eta2, logit replacement                  : 0.2383
ratio                                    : 1.86x  (threshold >= 2)

>>> P3 FAILS

secondary location summaries (reported, not counted toward P3):
mean_path_length       0.4704
err_pos_gini           0.4675
err_final_share        0.4308
err_late_layers        0.3335
error_mass_total       0.2418
err_pos_argmax_norm    0.1571

by category, primary measure:
                       mean    std
category                          
induction             0.335  0.050
obfuscated            0.436  0.061
factual_recall        0.440  0.036
code_completion       0.466  0.090
syntactic_agreement   0.468  0.058
arithmetic            0.489  0.036
multi_hop             0.497  0.031
entity_known_unknown  0.513  0.048


## Part 7 — P1

Spearman between the predicted category ranking (D3.6, 8 categories, multiple choice deleted) and
the ranking observed by mean replacement score.

In [80]:
PREDICTED = ["factual_recall", "syntactic_agreement", "arithmetic", "entity_known_unknown",
             "multi_hop", "code_completion", "induction", "obfuscated"]
observed = means["mean"].sort_values(ascending=False).index.tolist()
assert set(PREDICTED) == set(observed), set(PREDICTED) ^ set(observed)

pred_rank = {c: i for i, c in enumerate(PREDICTED)}
obs_rank  = {c: i for i, c in enumerate(observed)}
P1_RHO, P1_P = stats.spearmanr([pred_rank[c] for c in PREDICTED],
                               [obs_rank[c] for c in PREDICTED])
print(pd.DataFrame({"predicted": PREDICTED,
                    "observed": observed,
                    "mean_replacement": [round(means["mean"][c], 4) for c in observed]}).to_string(index=False))
print(f"\n>>> P1 Spearman = {P1_RHO:+.3f} (p = {P1_P:.3f})")

           predicted             observed  mean_replacement
      factual_recall            induction            0.7455
 syntactic_agreement entity_known_unknown            0.7423
          arithmetic  syntactic_agreement            0.7414
entity_known_unknown       factual_recall            0.7381
           multi_hop      code_completion            0.7357
     code_completion            multi_hop            0.7279
           induction           obfuscated            0.7273
          obfuscated           arithmetic            0.7092

>>> P1 Spearman = +0.071 (p = 0.867)


## Part 8 — P5 (secondary question)

*Replacement score and human rating correlate positively but weakly: |rho| < 0.4.*

Stream 1 only. Reported regardless of the kappa gate, but marked uninterpretable if the duplicate
kappa is below 0.6 (section 5).

In [28]:
rat = firsts["human"].rename("rating").reset_index()
J = rat.merge(df, on="prompt_id", how="left")
assert J.replacement_score.notna().all(), "a rated prompt is missing from results"

P5_RHO, P5_P = stats.spearmanr(J.replacement_score, J.rating)
print(f"n = {len(J)}   rating distribution: {J.rating.value_counts().reindex([0,1,2]).to_dict()}")
print(f">>> P5 Spearman(replacement, rating) = {P5_RHO:+.3f}  (p = {P5_P:.3f})")
print(f">>> prediction |rho| < 0.4: {'HOLDS' if abs(P5_RHO) < 0.4 else 'FAILS'}")
if HUMAN_KAPPA < 0.6:
    print(">>> NOTE: duplicate kappa below 0.6, so P5 is reported as UNINTERPRETABLE (section 5)")

print()
print(J.groupby("rating").agg(n=("rating", "size"),
                              mean_replacement=("replacement_score", "mean"),
                              mean_completeness=("completeness_score", "mean"),
                              mean_gini=("err_pos_gini_norm", "mean")).round(4).to_string())
print()
print(pd.crosstab(J.category, J.rating).to_string())

for other in [r for r in firsts if r != "human"]:
    o = firsts[other].rename("rating").reset_index().merge(df, on="prompt_id")
    rho, p = stats.spearmanr(o.replacement_score, o.rating)
    print(f"\n[exploratory] {other}: Spearman = {rho:+.3f} (p = {p:.3f}, n = {len(o)})")

n = 40   rating distribution: {0: 17, 1: 17, 2: 6}
>>> P5 Spearman(replacement, rating) = +0.243  (p = 0.132)
>>> prediction |rho| < 0.4: HOLDS

         n  mean_replacement  mean_completeness  mean_gini
rating                                                    
0       17            0.7313             0.9307     0.4365
1       17            0.7285             0.9307     0.4675
2        6            0.7551             0.9354     0.4523

rating                0  1  2
category                     
arithmetic            4  3  0
code_completion       0  3  0
entity_known_unknown  4  2  0
factual_recall        0  1  4
induction             5  0  0
multi_hop             1  2  2
obfuscated            2  3  0
syntactic_agreement   1  3  0

[exploratory] llm_opus: Spearman = +0.058 (p = 0.720, n = 40)

[exploratory] assistant: Spearman = +0.103 (p = 0.526, n = 40)


## Part 9 — Falsification (section 5)

Two rules, both pre-registered:

1. Category explains **under 10%** of variance in *every* outcome, and no predictor reaches
   significance after Benjamini–Hochberg → **no triage rule exists**.
2. Leave-one-category-out cross-validated R² is **at or below zero** for all outcomes → in-sample
   structure does not generalise to unseen prompt types, which is the actual use case.

In [81]:
OUTCOMES = {"replacement (logit)": logit(df.replacement_score),
            "completeness (logit)": logit(df.completeness_score),
            "error mass total": df.error_mass_total.values,
            "err_pos_gini_norm": df.err_pos_gini_norm.values,
            "err_final_share": df.err_final_share.values,
            "mean_path_length": df.mean_path_length.values}

eta_tbl = pd.Series({k: eta2(v, cat) for k, v in OUTCOMES.items()}).sort_values(ascending=False)
print("eta2 of category, by outcome:")
print(eta_tbl.round(4).to_string())
RULE1_ETA = bool((eta_tbl < 0.10).all())
print(f"\nrule 1, variance part: category explains <10% in every outcome -> {RULE1_ETA}")

eta2 of category, by outcome:
mean_path_length        0.4704
err_pos_gini_norm       0.4441
err_final_share         0.4308
error mass total        0.2418
replacement (logit)     0.2383
completeness (logit)    0.2099

rule 1, variance part: category explains <10% in every outcome -> False


In [30]:
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

PREDS = ["n_tok", "next_token_entropy", "top1_prob", "mean_token_logid"]
X_cont = sm.add_constant(df[PREDS].astype(float))

def loco_r2(y, use_category=False):
    # Leave-one-category-out CV R2, pooled over held-out predictions.
    # Category is deliberately NOT a predictor: the use case is a prompt type never seen,
    # for which no category coefficient exists.
    y = np.asarray(y, float)
    pred = np.full(len(y), np.nan)
    for held in df.category.unique():
        tr, te = (df.category != held).values, (df.category == held).values
        Xtr, Xte = X_cont[tr], X_cont[te]
        m = sm.OLS(y[tr], Xtr).fit()
        pred[te] = m.predict(Xte)
    ss_res = ((y - pred) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return float(1 - ss_res / ss_tot)

loco = pd.Series({k: loco_r2(v, False) for k, v in OUTCOMES.items()}).sort_values(ascending=False)
print("leave-one-category-out R2 (continuous predictors only, the usable rule):")
print(loco.round(4).to_string())
RULE2 = bool((loco <= 0).all())
print(f"\nrule 2: LOCO R2 <= 0 for all outcomes -> {RULE2}")

leave-one-category-out R2 (continuous predictors only, the usable rule):
err_final_share         0.2748
replacement (logit)     0.0669
error mass total        0.0532
completeness (logit)   -0.1431
err_pos_gini_norm      -0.1863
mean_path_length       -0.5348

rule 2: LOCO R2 <= 0 for all outcomes -> False


## Part 10 — The three models (6.1)

(a) category alone, (b) cheap continuous predictors alone, (c) both with variance inflation
factors. Model (b) is the one that matters practically: a practitioner has a prompt, not a category
label. VIF above 5 → (c) is reported for completeness but not used for inference.

In [82]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import patsy

y = logit(df.replacement_score)

ma = sm.OLS(y, patsy.dmatrix("C(category)", df, return_type="dataframe")).fit()
mb = sm.OLS(y, X_cont).fit()
mc = sm.OLS(y, patsy.dmatrix("C(category) + n_tok + next_token_entropy + top1_prob "
                             "+ mean_token_logid", df, return_type="dataframe")).fit()

print(f"(a) category only      R2 = {ma.rsquared:.4f}  adj = {ma.rsquared_adj:.4f}")
print(f"(b) continuous only    R2 = {mb.rsquared:.4f}  adj = {mb.rsquared_adj:.4f}")
print(f"(c) both               R2 = {mc.rsquared:.4f}  adj = {mc.rsquared_adj:.4f}")

Xv = patsy.dmatrix("C(category) + n_tok + next_token_entropy + top1_prob + mean_token_logid",
                   df, return_type="dataframe")
vifs = pd.Series({Xv.columns[i]: variance_inflation_factor(Xv.values, i)
                  for i in range(1, Xv.shape[1])}).sort_values(ascending=False)
print(f"\nmax VIF = {vifs.max():.1f}  -> model (c) "
      f"{'reported for completeness only, NOT used for inference' if vifs.max() > 5 else 'usable'}")
print(vifs.round(1).head(8).to_string())

print("\nmodel (b), the practically relevant one:")
print(pd.DataFrame({"coef": mb.params, "se": mb.bse, "p": mb.pvalues}).round(4).to_string())

(a) category only      R2 = 0.2383  adj = 0.2167
(b) continuous only    R2 = 0.2991  adj = 0.2879
(c) both               R2 = 0.4450  adj = 0.4199

max VIF = 23.1  -> model (c) reported for completeness only, NOT used for inference
next_token_entropy                     23.1
top1_prob                              18.3
C(category)[T.obfuscated]               6.4
C(category)[T.factual_recall]           4.3
C(category)[T.syntactic_agreement]      4.2
C(category)[T.multi_hop]                4.0
C(category)[T.entity_known_unknown]     3.8
C(category)[T.code_completion]          2.9

model (b), the practically relevant one:
                      coef      se       p
const               1.1966  0.0722  0.0000
n_tok               0.0178  0.0018  0.0000
next_token_entropy -0.0234  0.0142  0.0998
top1_prob          -0.1028  0.0685  0.1345
mean_token_logid   -0.0331  0.0054  0.0000


In [83]:
# Benjamini-Hochberg across the category contrasts and across outcomes
rows = []
for name, yv in OUTCOMES.items():
    m = sm.OLS(np.asarray(yv, float),
               patsy.dmatrix("C(category)", df, return_type="dataframe")).fit()
    for term, p in m.pvalues.items():
        if term.startswith("C(category)"):
            rows.append(dict(outcome=name, contrast=term.split("T.")[-1].rstrip("]"),
                             coef=m.params[term], p=p))
for name, yv in OUTCOMES.items():
    m = sm.OLS(np.asarray(yv, float), X_cont).fit()
    for term, p in m.pvalues.items():
        if term != "const":
            rows.append(dict(outcome=name, contrast=term, coef=m.params[term], p=p))

BH = pd.DataFrame(rows)
BH["p_adj"] = multipletests(BH.p, method="fdr_bh")[1]
BH["sig"] = BH.p_adj < 0.05
print(f"{BH.sig.sum()} of {len(BH)} contrasts significant after Benjamini-Hochberg")
print(BH.sort_values("p_adj").head(15).round(4).to_string(index=False))

RULE1 = RULE1_ETA and not BH.sig.any()
print(f"\nrule 1 (no triage rule exists): {RULE1}")
print(f"rule 2 (does not generalise)  : {RULE2}")
print(f"\n>>> NULL RESULT HEADLINE: {RULE1 or RULE2}")

47 of 66 contrasts significant after Benjamini-Hochberg
            outcome             contrast    coef   p  p_adj  sig
    err_final_share                n_tok -0.0215 0.0    0.0 True
  err_pos_gini_norm            induction -0.1542 0.0    0.0 True
    err_final_share            induction -0.2295 0.0    0.0 True
   mean_path_length       factual_recall -0.3194 0.0    0.0 True
replacement (logit)                n_tok  0.0178 0.0    0.0 True
   error mass total                n_tok -0.0034 0.0    0.0 True
   mean_path_length            multi_hop -0.2784 0.0    0.0 True
   mean_path_length            induction -0.2871 0.0    0.0 True
   mean_path_length                n_tok -0.0237 0.0    0.0 True
   error mass total            induction -0.0363 0.0    0.0 True
   mean_path_length  syntactic_agreement -0.2414 0.0    0.0 True
replacement (logit)            induction  0.1834 0.0    0.0 True
   error mass total entity_known_unknown -0.0331 0.0    0.0 True
replacement (logit) entity_known_u

In [84]:
summary = dict(
    n_graphs=int(len(df)),
    reliability={k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in REL.items()},
    pairwise=PAIRWISE.to_dict("records"),
    P1=dict(rho=round(float(P1_RHO), 3), p=round(float(P1_P), 4),
            predicted=PREDICTED, observed=observed),
    P2=dict(range=round(P2_RANGE, 4), eta2_logit=round(P2_ETA2, 4),
            eta2_raw=round(P2_ETA2_RAW, 4), holds=bool(P2)),
    P3=dict(eta2_gini_norm=round(P3_PRIMARY, 4), ratio=round(P3_PRIMARY / max(P2_ETA2, 1e-12), 2),
            holds=bool(P3), secondary={k: round(v, 4) for k, v in sec.items()}),
    P4=dict(range=round(P4_RANGE, 4), eta2=round(P4_ETA2, 4), holds=bool(P4)),
    P5=dict(rho=round(float(P5_RHO), 3), p=round(float(P5_P), 4),
            holds=bool(abs(P5_RHO) < 0.4), interpretable=bool(HUMAN_KAPPA >= 0.6),
            distribution=J.rating.value_counts().reindex([0, 1, 2]).to_dict()),
    falsification=dict(eta2_by_outcome={k: round(v, 4) for k, v in eta_tbl.items()},
                       loco_r2={k: round(v, 4) for k, v in loco.items()},
                       rule1=bool(RULE1), rule2=bool(RULE2), null_headline=bool(RULE1 or RULE2)),
    models=dict(category_only_r2=round(ma.rsquared, 4),
                continuous_only_r2=round(mb.rsquared, 4),
                both_r2=round(mc.rsquared, 4), max_vif=round(float(vifs.max()), 1)),
)
json.dump(summary, open(f"{OUT}/day6_confirmatory.json", "w"), indent=2, default=str)
J.to_csv(f"{OUT}/ratings_with_metrics.csv", index=False)
BH.to_csv(f"{OUT}/bh_contrasts.csv", index=False)
print(json.dumps({k: summary[k] for k in ["P1", "P2", "P3", "P4", "P5"]}, indent=2))
print("\nsaved day6_confirmatory.json -- DOWNLOAD /kaggle/working/out/")

{
  "P1": {
    "rho": 0.071,
    "p": 0.8665,
    "predicted": [
      "factual_recall",
      "syntactic_agreement",
      "arithmetic",
      "entity_known_unknown",
      "multi_hop",
      "code_completion",
      "induction",
      "obfuscated"
    ],
    "observed": [
      "induction",
      "entity_known_unknown",
      "syntactic_agreement",
      "factual_recall",
      "code_completion",
      "multi_hop",
      "obfuscated",
      "arithmetic"
    ]
  },
  "P2": {
    "range": 0.0363,
    "eta2_logit": 0.2383,
    "eta2_raw": 0.2418,
    "holds": false
  },
  "P3": {
    "eta2_gini_norm": 0.4441,
    "ratio": 1.86,
    "holds": false,
    "secondary": {
      "err_final_share": 0.4308,
      "err_pos_argmax_norm": 0.1571,
      "err_late_layers": 0.3335,
      "error_mass_total": 0.2418,
      "mean_path_length": 0.4704,
      "err_pos_gini": 0.4675
    }
  },
  "P4": {
    "range": 0.0079,
    "eta2": 0.2099,
    "holds": true
  },
  "P5": {
    "rho": 0.243,
    "p": 0.1

In [85]:
print(df[["err_final_share", "n_tok", "next_token_entropy",
          "top1_prob", "mean_token_logid"]].corr().round(3).to_string())

                    err_final_share  n_tok  next_token_entropy  top1_prob  mean_token_logid
err_final_share               1.000 -0.691               0.027     -0.111            -0.217
n_tok                        -0.691  1.000              -0.109      0.136             0.378
next_token_entropy            0.027 -0.109               1.000     -0.958            -0.116
top1_prob                    -0.111  0.136              -0.958      1.000             0.164
mean_token_logid             -0.217  0.378              -0.116      0.164             1.000


---

# EXPLORATORY — everything below is secondary

Listed here before any of it was run, so the set is fixed rather than assembled from whatever
looked interesting:

1. Direct token-to-logit share, parsed from the stored rating texts
2. `task_ok` subset, as registered and with the two flipped prompts (D3.4 outcome)
3. Rating by category; known vs unknown entity graphs
4. Frame length within item (Cuba, Peru: short vs long)
5. Where the rating streams disagree, by category
6. Stream 3 audit rates, including *defensible rating, unsupported reason*
7. Seconds per rating vs rating

In [86]:
# 1. direct token-to-logit share, from the trace text of the 40 rated graphs
import re
texts = {json.loads(l)["presentation"]: json.loads(l)
         for l in open(find_input("graph_texts.jsonl")) if l.strip()}

def direct_share(text):
    body = text[text.index("LOGIT "):] if "LOGIT " in text else text
    top = [l for l in body.split("\n")[1:] if l.startswith("  <-")]
    w = [float(re.search(r"(\d\.\d+)", l).group(1)) for l in top if re.search(r"\d\.\d+", l)]
    tok = [float(re.search(r"(\d\.\d+)", l).group(1)) for l in top
           if "TOKEN" in l and re.search(r"\d\.\d+", l)]
    return (sum(tok) / sum(w)) if w else np.nan

ds = pd.DataFrame([dict(presentation=n, prompt_id=t["prompt_id"],
                        direct_share=direct_share(t["text"])) for n, t in texts.items()])
ds = ds.drop_duplicates("prompt_id").merge(J[["prompt_id", "category", "rating",
                                              "replacement_score"]], on="prompt_id")
print("direct token-to-logit share of the logit's top inputs:")
print(ds.groupby("category").direct_share.agg(["mean", "min", "max", "size"]).round(3).to_string())
print(f"\noverall mean {ds.direct_share.mean():.3f}")
print(f"corr(direct share, replacement) = {ds.direct_share.corr(ds.replacement_score):+.3f}")
print(f"corr(direct share, rating)      = {ds.direct_share.corr(ds.rating):+.3f}")

direct token-to-logit share of the logit's top inputs:
                       mean    min    max  size
category                                       
arithmetic            0.215  0.000  0.322     7
code_completion       0.415  0.286  0.620     3
entity_known_unknown  0.297  0.213  0.433     6
factual_recall        0.615  0.541  0.654     5
induction             0.616  0.419  0.913     5
multi_hop             0.257  0.000  0.400     5
obfuscated            0.384  0.000  0.677     5
syntactic_agreement   0.246  0.000  0.421     4

overall mean 0.372
corr(direct share, replacement) = +0.556
corr(direct share, rating)      = +0.033


In [87]:
# 2. task_ok subset, as registered and with the two D3.4 flips
t = df.copy()
t["task_ok_bool"] = t.task_ok.astype(str).str.lower().map({"true": True, "false": False})
sub = t[t.task_ok_bool == True]
print(f"task_ok subset: {len(sub)} of {len(t)} prompts")
print(f"eta2 replacement, all prompts   : {P2_ETA2:.4f}")
print(f"eta2 replacement, task_ok subset: {eta2(logit(sub.replacement_score), sub.category.values):.4f}")

flip = t.copy()
flip.loc[flip.prompt_id.isin(["p0008", "p0117"]), "task_ok_bool"] = True
sub2 = flip[flip.task_ok_bool == True]
print(f"with p0008 and p0117 flipped    : "
      f"{eta2(logit(sub2.replacement_score), sub2.category.values):.4f}  (n={len(sub2)})")

task_ok subset: 131 of 255 prompts
eta2 replacement, all prompts   : 0.2383
eta2 replacement, task_ok subset: 0.3963
with p0008 and p0117 flipped    : 0.3882  (n=133)


In [89]:
print(sub.groupby("category").size().to_string())

category
arithmetic             27
factual_recall         32
induction              24
multi_hop              30
syntactic_agreement    18


In [90]:
cats_with_task = sub.category.unique()
same_cats = df[df.category.isin(cats_with_task)]
print(f"{len(cats_with_task)} categories")
print(f"eta2, those categories, all prompts : {eta2(logit(same_cats.replacement_score), same_cats.category.values):.4f}")
print(f"eta2, those categories, task_ok only: {eta2(logit(sub.replacement_score), sub.category.values):.4f}")

5 categories
eta2, those categories, all prompts : 0.2859
eta2, those categories, task_ok only: 0.3963


In [91]:
# 3. rating by category; known vs unknown entity
print(pd.crosstab(J.category, J.rating, margins=True).to_string())
ent = J[J.category == "entity_known_unknown"].copy()
if len(ent):
    ent["arm"] = ent.subtype.str.split("|").str[0]
    print("\nentity arms among rated graphs:")
    print(ent.groupby("arm").agg(n=("rating", "size"), mean_rating=("rating", "mean"),
                                 mean_replacement=("replacement_score", "mean")).round(3).to_string())

rating                 0   1  2  All
category                            
arithmetic             4   3  0    7
code_completion        0   3  0    3
entity_known_unknown   4   2  0    6
factual_recall         0   1  4    5
induction              5   0  0    5
multi_hop              1   2  2    5
obfuscated             2   3  0    5
syntactic_agreement    1   3  0    4
All                   17  17  6   40

entity arms among rated graphs:
         n  mean_rating  mean_replacement
arm                                      
known    3        0.000             0.742
unknown  3        0.667             0.745


In [92]:
# 4. frame length within item
fr = J[J.category.isin(["factual_recall", "multi_hop"])].copy()
fr["frame"] = np.where(fr.subtype.str.contains("long"), "long", "short")
print(fr.groupby("frame").agg(n=("rating", "size"), mean_rating=("rating", "mean"),
                              mean_replacement=("replacement_score", "mean"),
                              mean_n_tok=("n_tok", "mean")).round(3).to_string())
print("\nnote: short/long also differ in format (FACT_PREFIX on short frames only, D2.3)")

       n  mean_rating  mean_replacement  mean_n_tok
frame                                              
long   5          1.6             0.758        14.2
short  5          1.4             0.728         9.0

note: short/long also differ in format (FACT_PREFIX on short frames only, D2.3)


In [93]:
fr = df[df.category.isin(["factual_recall", "multi_hop"])].copy()
fr["frame"] = np.where(fr.subtype.str.contains("long"), "long", "short")
fr["item"] = fr.subtype.str.split("|").str[-1]
piv = fr.pivot_table(index="item", columns="frame", values="replacement_score")
piv = piv.dropna()
print(f"{len(piv)} matched items")
print(f"mean within-item difference (long - short): {(piv['long'] - piv['short']).mean():+.4f}")
print(stats.wilcoxon(piv["long"], piv["short"]))

0 matched items
mean within-item difference (long - short): +nan
WilcoxonResult(statistic=np.float64(nan), pvalue=np.float64(nan))


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1006: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)


In [ ]:
# 5. where the streams disagree, by category
if len(firsts) > 1:
    cmp = pd.DataFrame(firsts).join(df.set_index("prompt_id")[["category"]])
    cmp = cmp.dropna()
    for a, b in itertools.combinations([c for c in cmp.columns if c != "category"], 2):
        d = cmp[cmp[a] != cmp[b]]
        print(f"\n{a} vs {b}: {len(d)} disagreements of {len(cmp)}")
        if len(d):
            print(d.groupby("category").size().to_string())

In [94]:
fr = df[df.category.isin(["factual_recall", "multi_hop"])]
print(fr.subtype.value_counts().to_string())
print(fr[["prompt_id", "subtype", "expected", "n_tok"]].head(12).to_string(index=False))

subtype
capital_short         12
capital_med           12
capital_long          12
state_capital_long    10
state_capital         10
city_language          7
city_language_long     5
sport_short            4


KeyError: "['expected'] not in index"

In [96]:
fr = df[df.category.isin(["factual_recall", "multi_hop"])].copy()
fr = fr.merge(corpus[["prompt_id", "expected"]], on="prompt_id", how="left", suffixes=("", "_e"))
ecol = "expected" if "expected" in fr.columns else "expected_e"
fr["frame"] = np.where(fr.subtype.str.contains("long"), "long",
                np.where(fr.subtype.str.contains("med"), "med", "short"))

piv = fr.pivot_table(index=ecol, columns="frame", values="replacement_score", aggfunc="mean")
pair = piv[["short", "long"]].dropna()
print(f"{len(pair)} items with both a short and a long frame")
if len(pair) >= 6:
    d

27 items with both a short and a long frame


In [97]:
d = pair["long"] - pair["short"]
print(f"mean difference (long - short): {d.mean():+.4f}")
print(f"median: {d.median():+.4f}   positive: {(d > 0).sum()}/{len(d)}")
print(stats.wilcoxon(pair["long"], pair["short"]))

mean difference (long - short): +0.0323
median: +0.0196   positive: 27/27
WilcoxonResult(statistic=np.float64(0.0), pvalue=np.float64(1.4901161193847656e-08))


In [98]:
tri = piv[["short", "med", "long"]].dropna()
print(f"\n{len(tri)} items with all three frames")
print(tri.mean().round(4).to_string())
print(stats.friedmanchisquare(tri["short"], tri["med"], tri["long"]))


12 items with all three frames
frame
short    0.7147
med      0.7378
long     0.7703
FriedmanchisquareResult(statistic=np.float64(24.0), pvalue=np.float64(6.144212353328207e-06))


In [95]:
# 6. audit rates (stream 3)
if audit is not None and len(audit):
    au = audit.dropna(subset=["reason_supported"]).copy()
    for col in ["reason_supported", "rating_follows"]:
        au[col] = au[col].astype(str).str.lower().map({"true": True, "false": False})
    AUD = dict(n=int(len(au)),
               reason_unsupported=float((~au.reason_supported).mean()),
               rating_does_not_follow=float((~au.rating_follows).mean()),
               defensible_rating_unsupported_reason=float((au.rating_follows & ~au.reason_supported).mean()),
               judge_rating_differs=float((au.your_rating.astype(int) != au.human_rating.astype(int)).mean()))
    print(json.dumps(AUD, indent=2))
    print("\nexamples judged unsupported:")
    print(au.loc[~au.reason_supported, ["presentation", "human_rating", "comment"]].head(8).to_string(index=False))
    json.dump(AUD, open(f"{OUT}/day6_audit_rates.json", "w"), indent=2)

{
  "n": 50,
  "reason_unsupported": 0.16,
  "rating_does_not_follow": 0.18,
  "defensible_rating_unsupported_reason": 0.12,
  "judge_rating_differs": 0.3
}

examples judged unsupported:
 presentation  human_rating                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              comment
            2             1 The vague label the reason cites is the read-out feature (truncated, and plausibly 'relations between the US and Russia', with #760 explicitly naming Russia), while the intermediate at the content position, L19/L14 pos5 «mentions of Russia...», is clearly named, is not er

In [99]:
# 7. time per rating vs rating
h = STREAMS["human"].drop_duplicates("presentation")
print(h.groupby("rating").seconds.agg(["count", "median", "mean", "max"]).round(1).to_string())
rho_t, p_t = stats.spearmanr(h.rating, h.seconds)
print(f"\nSpearman(rating, seconds) = {rho_t:+.3f} (p = {p_t:.3f})")
print(f"over the 5-minute cap: {(h.seconds > 300).sum()} of {len(h)}")

        count  median   mean    max
rating                             
0          22   251.8  252.2  402.4
1          21   284.1  322.0  682.9
2           7   276.6  324.4  445.2

Spearman(rating, seconds) = +0.303 (p = 0.033)
over the 5-minute cap: 18 of 50


## Record in the pre-registration

> **Confirmatory outcomes [DATE].** P1 Spearman [__]. P2 range [__], eta² [__] → [HOLDS/FAILS].
> P3 eta²(normalised interior Gini) [__] vs eta²(logit replacement) [__], ratio [__] →
> [HOLDS/FAILS]. P4 completeness range [__] vs replacement [__] → [HOLDS/FAILS]. P5 Spearman [__]
> (n = 40), duplicate kappa [__] → [interpretable / uninterpretable], prediction [HOLDS/FAILS].
>
> **Falsification.** Category eta² by outcome [__]; leave-one-category-out R² [__]; rule 1 [__],
> rule 2 [__]. Headline is [a null result / a positive finding].
>
> **Models.** (a) category R² [__], (b) continuous R² [__], (c) both R² [__], max VIF [__].
> [__] of [__] contrasts significant after Benjamini–Hochberg.

In [100]:
ent = STREAMS["human"].drop_duplicates("presentation").merge(
    df[["prompt_id", "category"]], on="prompt_id")
e = ent[ent.category == "entity_known_unknown"].sort_values("presentation")
print(e[["presentation", "prompt_id", "rating"]].to_string(index=False))

 presentation prompt_id  rating
            1     p0204       1
            4     p0205       0
           20     p0186       0
           22     p0198       1
           26     p0195       0
           32     p0185       0
           45     p0186       1
           48     p0198       1


In [101]:
alt = firsts["human"].copy(); alt["p0204"] = 0
j2 = alt.rename("rating").reset_index().merge(df, on="prompt_id")
print(stats.spearmanr(j2.replacement_score, j2.rating))

SignificanceResult(statistic=np.float64(0.21652394513287518), pvalue=np.float64(0.17960354602119502))


Save the numbers that aren't in any file

Several results came from cells you ran ad hoc, and they exist only in the notebook output:

In [102]:
extras = dict(
    matched_frames=dict(n_pairs=27, mean_diff=0.0323, median_diff=0.0196,
                        positive=27, wilcoxon_p=1.49e-8,
                        three_frame_means=[0.7147, 0.7378, 0.7703],
                        friedman_p=6.14e-6, n_items_three=12),
    direct_path=dict(overall_mean=0.372, corr_replacement=0.556, corr_rating=0.033,
                     by_category={"induction": 0.616, "factual_recall": 0.615,
                                  "code_completion": 0.415, "obfuscated": 0.384,
                                  "entity_known_unknown": 0.297, "multi_hop": 0.257,
                                  "syntactic_agreement": 0.246, "arithmetic": 0.215}),
    task_ok=dict(all_prompts=0.2383, subset=0.3963, same_five_cats_all=0.2859, flipped=0.3882),
    err_final_share_vs_ntok=-0.691,
    entropy_vs_top1prob=-0.958,
    p5_sensitivity=dict(recorded=0.243, presentation1_as_zero=0.217,
                        p_recorded=0.132, p_alt=0.180),
    timing=dict(spearman_rating_seconds=0.303, p=0.033, over_cap=18,
                interruptions={"27": 150, "33": 60, "37": 420}),
)
json.dump(extras, open(f"{OUT}/day6_exploratory.json", "w"), indent=2)
print(json.dumps(extras, indent=2)[:600])

{
  "matched_frames": {
    "n_pairs": 27,
    "mean_diff": 0.0323,
    "median_diff": 0.0196,
    "positive": 27,
    "wilcoxon_p": 1.49e-08,
    "three_frame_means": [
      0.7147,
      0.7378,
      0.7703
    ],
    "friedman_p": 6.14e-06,
    "n_items_three": 12
  },
  "direct_path": {
    "overall_mean": 0.372,
    "corr_replacement": 0.556,
    "corr_rating": 0.033,
    "by_category": {
      "induction": 0.616,
      "factual_recall": 0.615,
      "code_completion": 0.415,
      "obfuscated": 0.384,
      "entity_known_unknown": 0.297,
      "multi_hop": 0.257,
      "syntactic_agree
